# Pooled-sample subtyping: are the per-sample subtypes genuine?

`3_clustering.ipynb` subtypes the tumor cells of each Xenium 5K sample on its own. This notebook pools the tumor
cells of the five samples (BC, OC, CC, LC, Prostate; ~624k cells) into **one** object, processes and analyses it
with **exactly the per-sample procedure** (same gene set, depth correction, clustering, program dictionary,
annotation rule and figures, all from `subtype_utils.py`), and asks three questions:

1. **Existence.** Do the subtypes identified per sample reappear when all cells are clustered together?
2. **Correspondence.** Within each sample, do the per-sample labels and the pooled labels agree cell by cell?
3. **Batch effect.** Do the five samples mix in the pooled embedding (UMAP / t-SNE, kNN neighbourhoods, PCs), i.e.
   is the depth-corrected cytoplasmic expression of the target genes free of a strong sample effect?

**Versions.** This notebook is the uncorrected pool (`CORRECTION = "none"` in the builder: one depth regression and one
scaling over all cells, sample means left in; result 2026-11-03: the samples separate completely, no per-sample subtype
recovered). Its sibling `4_pooled_sample_corrected.ipynb` runs the same analysis on the sample-corrected pool
(`output/1_exploration/pooled_sample_corrected/`: depth regression and scaling within each sample, then pooled PCA);
both are generated from the same builder and keep their own output folder and reviewed config.

If the answers are yes, the subtypes are cross-sample substates of the cytoplasmic program rather than per-sample
artifacts, and a pooled label set can serve as the outcome vocabulary of later stages.

**Inputs**
- `output/1_exploration/pooled/adata_pooled_cytoplasmic_embedded.h5ad`: all tumor cells x gene set, built on HGCC by
  `4_pooled_embedding.py` (per sample: normalize_total on the full panel, gene subset, log1p in `layers["lognorm"]`;
  pooled: depth regressed out, scaled, PCA, kNN graph, UMAP, t-SNE; no correction for sample).
- `output/1_exploration/<sample>/characteristics/cell_annotations_cytoplasmic.csv` and
  `annotation/cell_subtypes_cytoplasmic.csv`: the per-sample labels and per-cell characteristics (spatial context,
  ssGSEA and 8-TF hypoxia, morphology) of `3_clustering.ipynb`, joined on `sample` + `cell_id`.
- `data/<sample>/intermediate_data/adata.h5ad` (obs only, via h5py): coordinates and cell types of all cells, for the
  tissue-map backgrounds.
- `data/_utils/*.gmt`: the program dictionary.

**Outputs** (`output/1_exploration/pooled/`; the subfolders are emptied on every run, the `.h5ad` in the root is kept)

| folder | content |
|---|---|
| `clustering/` | resolution sweep, cluster statistics, sample composition per cluster / label, UMAPs (cluster, label, sample), PCA loadings, tiled tissue map by label, label composition |
| `DE/` | one marker table per pooled cluster, marker dotplot by label |
| `annotation/` | pooled cluster -> label table, per-cell record `cell_subtypes_pooled.csv` (pooled and per-sample labels), program-score heatmap |
| `spatial/` | one tiled tissue map per pooled label (alone, on immune cells, on T cells) |
| `characteristics/` | per-label summary and per-cell tables, box plots, tiled maps of scores and proximities |
| `comparison/` | canvas layout, label presence, correspondence tables / heatmaps / scores, label agreement, batch-mixing tables and plots, summary |

**Saved results for later stages**: `annotation/subtype_annotation_cytoplasmic.csv`, `annotation/cell_subtypes_pooled.csv`
and the pooled `.h5ad` rewritten with the pooled labels, program scores and canvas coordinates in `obs`.

**How the pooled clustering is reviewed**: as for a sample. Run with the sweep's suggestion and no manual labels, review
`clustering/resolution_sweep_cytoplasmic.csv`, `annotation/subtype_annotation_cytoplasmic.csv`,
`clustering/cluster_stats_cytoplasmic.csv`, `clustering/cluster_sample_composition.csv` and the `DE/` tables, record the
decisions in `POOLED_CONFIG` (dated comment) and run again.

## 1. Setup

Imports, paths and the pooled configuration. Parameters, program dictionary, label vocabulary and all analysis
functions come from `subtype_utils.py`, shared with `3_clustering.ipynb`. The six output subfolders are emptied and
recreated on every run.

In [1]:
import datetime
import gc
import os
import shutil

import h5py
import matplotlib.colors as clr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy import sparse
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

# parameters, colours, program dictionary, label vocabulary and the analysis functions shared with 3_clustering.ipynb
from subtype_utils import *

set_figsize((8, 8))

/Users/chenyang/miniconda3/envs/AIVC-env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/Users/chenyang/miniconda3/envs/AIVC-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/AIVC-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/AIVC-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `annda

In [2]:
utils_path = "../../data/_utils/"
pooled_path = "../../output/1_exploration/pooled/"
sample_output = lambda sample: f"../../output/1_exploration/{sample}/"

paths = {"output": pooled_path}
for name in ["clustering", "DE", "annotation", "spatial", "characteristics", "comparison"]:
    paths[name] = pooled_path + name + "/"
    shutil.rmtree(paths[name], ignore_errors=True)
    os.makedirs(paths[name], exist_ok=True)

In [3]:
SAMPLES = ["Xenium_5K_BC", "Xenium_5K_OC", "Xenium_5K_CC", "Xenium_5K_LC", "Xenium_5K_Prostate"]
short = lambda sample: sample.replace("Xenium_5K_", "")

# Tissue-map canvas: the samples' coordinates are shifted so that the five tissues sit side by side on one canvas,
# three in the top row and two in the bottom row, with a gap of CANVAS_GAP x the row width between tissues.
CANVAS_LAYOUT = [["Xenium_5K_BC", "Xenium_5K_OC", "Xenium_5K_CC"], ["Xenium_5K_LC", "Xenium_5K_Prostate"]]
CANVAS_GAP = 0.03
CANVAS_WIDTH_IN = 18   # figure width of the tiled tissue maps; the height follows the canvas aspect ratio

# Pooled clustering configuration, reviewed like a sample's entry in SAMPLE_CONFIG (3_clustering.ipynb):
# - res_cyto: Louvain resolution (the sweep in section 4 prints a suggestion; this is the reviewed value)
# - manual_labels: reviewed label decisions (pooled cluster id -> label), applied after the automatic annotation
# Reviewed 2026-11-03 (first pass at the sweep suggestion 1.0, 16 clusters). The pooled clustering follows the samples:
# every cluster except 9 (Interferon), 12 (Myeloid border) and the Low depth clusters is >= 85 % one sample, and each
# sample's bulk falls into one or two clusters (OC 0 + 6, BC 1 + 11, Prostate 2 + 10, CC 3 / 4 / 8, LC 7). Higher
# resolutions (1.2, 1.5) only subdivide within samples (add G2/M, min cluster 32 at 1.2), so 1.0 is kept.
# Label decisions, with the per-sample conventions of SAMPLE_CONFIG:
# - 0 -> Baseline: OC bulk (99 % OC, 66 % of OC cells); markers H19, SULF1, IMPDH2, LAPTM4B, EEF1G are the OC identity
#   genes of the per-sample Baseline clusters; "Oxidative stress" came from the score criterion only (z 0.63), the
#   Stress response overlap is not significant (p 0.02).
# - 1 -> Baseline: BC bulk (98 % BC, 87 % of BC cells); XBP1, TSPAN13, CCND1, FOXA1, RARA are luminal identity genes
#   (BC Baseline clusters 0 / 1 per sample); "ER stress" by score only (z 0.77), Apoptosis overlap p 0.04.
# - 2 -> Baseline (confirmed): Prostate bulk (98 %); NKX3-1, SORD, FOXA1, SPON2 identity, no significant overlap.
# - 7 -> Baseline: the whole LC sample (99.6 % LC, 92 % of LC cells) with 413 markers of lung identity (EPDR1, HOPX,
#   SLC40A1, CDH1, EPAS1); the Autophagy / Lysosome call is the long-marker-list effect noted for LC per sample.
# - 8 -> Stress response: CC cluster with p53 overlap (CDKN1A, GADD45A, SAMD9, BCL2L1, TGFA) and TNFA/NFKB score
#   (z 1.0); 77 % of its cells were Stress response per sample (p53 / TNFA/NFKB -> Stress response convention).
# Kept as annotated: 3 Hypoxia (LDHA, NDRG1, SLC2A1), 4 and 6 S-phase (6 merges the OC S-phase and G2/M cells),
# 9 Interferon (CXCL10, CXCL9, GBP1, STAT1; the only cluster shared by samples), 10 Stroma border (TGFB3, CCN2,
# SFRP1, DKK3 with median fibroblast distance 16 um; these Prostate cells were Stress response per sample),
# 11 Complement / acute phase (C4A, C4B, SERPINA3, CFB), 12 Myeloid border (CD68, ITGB2, SIRPA), 13 Stroma border
# (POSTN, COL5A1, THY1), 5 / 14 / 15 Low depth.
POOLED_CONFIG = {"res_cyto": 1.0,
                 "manual_labels": {"0": "Baseline", "1": "Baseline", "2": "Baseline", "7": "Baseline",
                                   "8": "Stress response"}}

## 2. Load the pooled object, the per-sample results and the tissue canvas

The pooled object carries, per cell, the sample and the metadata kept by `4_pooled_embedding.py`. The per-sample
results of `3_clustering.ipynb` are joined on `sample` + `cell_id`: the per-sample cluster and label
(`subtype_louvain_sample`, `subtype_label_sample`, `confidence_sample`) and the per-cell characteristics that depend
only on the sample (distances / proximities to the reference cell groups, neighbourhood composition, ssGSEA and 8-TF
hypoxia scores, morphology). They are identical to what a recomputation per sample would give, so they are taken from
the tables rather than recomputed; the program scores, depth statistics, clusters and labels are recomputed on the
pooled object. Every pooled cell must be found in the per-sample tables.

For the tissue maps the samples' coordinates are shifted onto one canvas (`canvas_x`, `canvas_y`): BC, OC, CC in
the top row, LC, Prostate in the bottom row (`CANVAS_LAYOUT`), written to `comparison/canvas_layout.csv`. The
backgrounds of the per-label maps (immune cells, T cells) come from the obs of all cells of each sample.

In [4]:
adata = sc.read_h5ad(pooled_path + "adata_pooled_cytoplasmic_embedded.h5ad")
adata.obs["sample"] = pd.Categorical(adata.obs["sample"].astype(str), categories=SAMPLES)
adata.obs["cell_id"] = adata.obs["cell_id"].astype(str)
adata.obs.drop(columns=[c for c in adata.obs.columns if c.startswith("louvain_") or c.startswith("subtype_") or c.startswith("score_")],
               inplace=True, errors="ignore")   # columns of a previous run of this notebook (the .h5ad is rewritten in section 8)
print(f"{adata.shape[0]} tumor cells x {adata.shape[1]} genes; per sample: {adata.obs['sample'].value_counts().loc[SAMPLES].to_dict()}")
print(f"embedding: {adata.uns['embedding_params']}")
assert "X_umap" in adata.obsm and "distances" in adata.obsp, "the pooled object must carry the kNN graph and the UMAP"

# per-sample results of 3_clustering.ipynb, joined on sample + cell_id
spatial_cols = [f"dist_{name}" for name in spatial_refs] + [f"proximity_{name}" for name in spatial_refs] + \
               [f"frac_immune_{RADIUS}um", f"frac_tcell_{RADIUS}um", f"n_cells_{RADIUS}um"]
char_cols = ["hypoxia_ssgsea_cyto", "hypoxia_score_tf", "interferon_ssgsea_cyto", "nucleus_cell_area_ratio"]
per_sample = []
for s in SAMPLES:
    chars = pd.read_csv(sample_output(s) + "characteristics/cell_annotations_cytoplasmic.csv", dtype={"cell_id": str})
    subtypes = pd.read_csv(sample_output(s) + "annotation/cell_subtypes_cytoplasmic.csv", dtype={"cell_id": str})
    assert (chars["cell_id"].values == subtypes["cell_id"].values).all()
    df = chars[["cell_id"] + spatial_cols + char_cols].copy()
    df["subtype_louvain_sample"] = subtypes["subtype_louvain_cyto"].astype(str).values
    df["subtype_label_sample"] = subtypes["subtype_label"].values
    df["confidence_sample"] = subtypes["confidence"].values
    df["resolution_sample"] = subtypes["resolution"].values
    df.index = s + ":" + df["cell_id"].values
    per_sample.append(df.drop(columns="cell_id"))
per_sample = pd.concat(per_sample)
missing = adata.obs_names.difference(per_sample.index)
assert len(missing) == 0, f"{len(missing)} pooled cells are missing from the per-sample tables"
per_sample = per_sample.loc[adata.obs_names]
for col in per_sample.columns:
    adata.obs[col] = per_sample[col].values
sample_labels = [label for label in LABEL_ORDER if label in set(adata.obs["subtype_label_sample"])]
adata.obs["subtype_label_sample"] = pd.Categorical(adata.obs["subtype_label_sample"], categories=sample_labels)
print(f"per-sample labels joined: {len(sample_labels)} labels; "
      + ", ".join(f"{short(s)} {adata.obs.loc[adata.obs['sample'] == s, 'subtype_label_sample'].nunique()}" for s in SAMPLES))
del per_sample

# all cells of every sample (obs only): cell types and coordinates for the tissue-map backgrounds
obs_all = []
for s in SAMPLES:
    with h5py.File(f"../../data/{s}/intermediate_data/adata.h5ad", "r") as f:
        group = f["obs"]["cell_type_merged"]
        cell_types = group["categories"][:].astype(str)[group["codes"][:]]
        df = pd.DataFrame({"sample": s, "cell_type_merged": cell_types,
                           "global_x": f["obs"]["global_x"][:], "global_y": f["obs"]["global_y"][:]})
    obs_all.append(df)
obs_all = pd.concat(obs_all, ignore_index=True)
print(f"{obs_all.shape[0]} cells of all types in the five samples")

# canvas: shift every sample to its own tile (rows of CANVAS_LAYOUT, top row first)
extent = obs_all.groupby("sample")[["global_x", "global_y"]].agg(["min", "max"])
layout = pd.DataFrame({"x_min": extent[("global_x", "min")], "y_min": extent[("global_y", "min")],
                       "width": extent[("global_x", "max")] - extent[("global_x", "min")],
                       "height": extent[("global_y", "max")] - extent[("global_y", "min")]}).loc[SAMPLES]
row_width = max(layout.loc[row, "width"].sum() for row in CANVAS_LAYOUT)
gap = CANVAS_GAP * row_width
row_heights = [layout.loc[row, "height"].max() for row in CANVAS_LAYOUT]
y_off = 0.0
for row, row_height in zip(reversed(CANVAS_LAYOUT), reversed(row_heights)):   # bottom row at y = 0
    x_off = 0.0
    for s in row:
        layout.loc[s, "x_offset"], layout.loc[s, "y_offset"] = x_off, y_off
        layout.loc[s, "row"] = CANVAS_LAYOUT.index(row)
        x_off += layout.loc[s, "width"] + gap
    y_off += row_height + gap
canvas_w = max(layout.loc[row, "width"].sum() + gap * (len(row) - 1) for row in CANVAS_LAYOUT)
canvas_h = sum(row_heights) + gap * (len(CANVAS_LAYOUT) - 1)
fig_size = (CANVAS_WIDTH_IN, CANVAS_WIDTH_IN * canvas_h / canvas_w)
layout.round(1).to_csv(paths["comparison"] + "canvas_layout.csv")

def to_canvas(df):
    s = df["sample"].astype(str).values
    df["canvas_x"] = df["global_x"].values - layout.loc[s, "x_min"].values + layout.loc[s, "x_offset"].values
    df["canvas_y"] = df["global_y"].values - layout.loc[s, "y_min"].values + layout.loc[s, "y_offset"].values

to_canvas(obs_all)
to_canvas(adata.obs)

def label_canvas(ax):
    """Sample names at the top-left corner of each tile."""
    for s in SAMPLES:
        ax.text(layout.loc[s, "x_offset"], layout.loc[s, "y_offset"] + layout.loc[s, "height"], short(s),
                ha="left", va="bottom", fontsize=11, fontweight="bold")

print(f"canvas {canvas_w:.0f} x {canvas_h:.0f} um, tissue maps at {fig_size[0]:.0f} x {fig_size[1]:.1f} in")
layout.round(0)

623838 tumor cells x 634 genes; per sample: {'Xenium_5K_BC': 102180, 'Xenium_5K_OC': 160250, 'Xenium_5K_CC': 221355, 'Xenium_5K_LC': 44624, 'Xenium_5K_Prostate': 95429}
embedding: {'depth_correction': 'pooled OLS residuals on log10(compartment_total_counts + 1) + scale(max_value=10); no correction for sample', 'gene_set_file': 'overlap_genes_3plus_samples.npy', 'n_cells_per_sample': {'Xenium_5K_BC': 102180, 'Xenium_5K_CC': 221355, 'Xenium_5K_LC': 44624, 'Xenium_5K_OC': 160250, 'Xenium_5K_Prostate': 95429}, 'n_comps_pca': 50, 'n_genes': 634, 'n_neighbors': 30, 'n_pcs': 30, 'normalization': "per sample: normalize_total(1e4) on full panel before gene subset, then log1p (layers['lognorm'])", 'samples': array(['Xenium_5K_BC', 'Xenium_5K_OC', 'Xenium_5K_CC', 'Xenium_5K_LC',
       'Xenium_5K_Prostate'], dtype=object), 'tsne': True}


per-sample labels joined: 15 labels; BC 10, OC 8, CC 11, LC 10, Prostate 9


1790452 cells of all types in the five samples


canvas 34846 x 24296 um, tissue maps at 18 x 12.6 in


,x_min,y_min,width,height,x_offset,y_offset,row
sample,,,,,,,
Xenium_5K_BC,261.0,6.0,9934.0,15504.0,0.0,8792.0,0.0
Xenium_5K_OC,13.0,22.0,11468.0,7955.0,10920.0,8792.0,0.0
Xenium_5K_CC,6.0,646.0,11471.0,10197.0,23375.0,8792.0,0.0
Xenium_5K_LC,9.0,107.0,11482.0,7806.0,0.0,0.0,1.0
Xenium_5K_Prostate,8.0,28.0,11498.0,6420.0,12469.0,0.0,1.0


## 3. Program scores (`build_programs`, `score_programs`)

The same program dictionary as per sample, restricted to the gene set; every cell is scored for every program on its
log-normalized cytoplasmic expression and the scores are z-scored over all pooled cells (so a z of 0.5 means the same
thing in every sample).

In [5]:
programs_raw = load_programs_raw(utils_path)
programs, label_programs = build_programs(set(adata.var_names), programs_raw)
scores_z = score_programs(adata, adata, programs)

S-phase                       47 genes: ANP32E, BIRC5, CBX5, CCNB2, CCNE1, CDC20, CDKN1A, CDKN2A, CDKN3, CHEK1, ...
G2/M                          40 genes: BIRC5, BUB1, CCNA2, CCNB2, CCND1, CDC20, CDKN3, CENPF, CHEK1, CKS2, ...
Interferon                    15 genes: IFIT1, IFIT2, IFIT3, IFI35, IFIH1, STAT1, IFITM1, RSAD2, EIF2AK2, GBP1, ...
Complement / acute phase       7 genes: SERPINA3, CFB, C4A, C4B, UBD, CFH, CD59
Stress response               10 genes: HSPA8, HSPB8, DNAJA1, BAG3, SQSTM1, KLF4, CDKN1A, GADD45A, ZFAND5, SERPINH1
Lysosome                      10 genes: CTSL, LGMN, LIPA, GRN, CREG1, GLB1, GLA, MAN2B1, LAPTM4B, GNS
Cholesterol synthesis          6 genes: HMGCR, SQLE, DHCR24, FDFT1, ACLY, EBP
Hypoxia                       26 genes: ALDOA, ANXA2, CCN2, CDKN1A, CDKN1C, COL5A1, CXCR4, FBP1, HK2, LDHA, ...
Glycolysis                    22 genes: ALDOA, CD44, COL5A1, CXCR4, FUT8, GFPT1, GOT1, HK2, HMMR, IDH1, ...
EMT/stroma                    22 genes: CCN2, CD44, CD59, CD

## 4. Resolution sweep and clustering (`resolution_sweep`, `cluster`)

Louvain on the pooled kNN graph (30 PCs, 30 neighbours, built on HGCC) at the resolutions of `RES_GRID`; the sweep
reports how many programs the score criterion resolves at each value and suggests the coarsest resolution that
resolves all of them. The clustering uses `POOLED_CONFIG["res_cyto"]`, the reviewed value.

In [6]:
sweep = resolution_sweep(adata, scores_z, label_programs, POOLED_CONFIG, paths)
cluster(adata, adata, POOLED_CONFIG["res_cyto"])

 resolution  n_clusters  min_cluster_size  n_program_labels                                                                                                                                   programs
        0.5          10               181                 6                                                                     EMT/stroma, ER stress, Hypoxia, Interferon, Lysosome, Oxidative stress
        0.8          14               181                 9                                 EMT/stroma, ER stress, Hypoxia, Interferon, Lysosome, Myeloid border, Oxidative stress, S-phase, TNFA/NFKB
        1.0          16               181                10       Complement / acute phase, EMT/stroma, ER stress, Hypoxia, Interferon, Lysosome, Myeloid border, Oxidative stress, S-phase, TNFA/NFKB
        1.2          17                32                11 Complement / acute phase, EMT/stroma, ER stress, G2/M, Hypoxia, Interferon, Lysosome, Myeloid border, Oxidative stress, S-phase, TNFA/NFKB
     

Number of clusters (cytoplasmic, resolution 1.0): 16


## 5. Cluster statistics and sample composition (`cluster_statistics`)

Depth check of the pooled embedding (Spearman of PC1-5 with log depth), per-cluster depth, target-gene counts, area
and distances (the inputs of the annotation overrides), and, specific to the pooled analysis, the **sample
composition of every cluster**: the share of each sample among the cluster's cells next to the sample's share of all
cells. `top_sample_ratio` = share of the dominant sample / its expected share; a cluster with a ratio near
1 / (expected share) consists of one sample only and is a candidate sample-specific state (or a batch artifact).

In [7]:
cluster_stats = cluster_statistics(adata, adata, paths)

expected = adata.obs["sample"].value_counts(normalize=True).loc[SAMPLES]
composition_cl = pd.crosstab(adata.obs["subtype_louvain_cyto"], adata.obs["sample"], normalize="index")[SAMPLES]
composition_cl["n_cells"] = adata.obs["subtype_louvain_cyto"].value_counts()
composition_cl["top_sample"] = composition_cl[SAMPLES].idxmax(axis=1)
composition_cl["top_share"] = composition_cl[SAMPLES].max(axis=1)
composition_cl["top_sample_ratio"] = composition_cl["top_share"] / expected.loc[composition_cl["top_sample"]].values
composition_cl.round(3).to_csv(paths["clustering"] + "cluster_sample_composition.csv")
print("expected shares: " + ", ".join(f"{short(s)} {v:.2f}" for s, v in expected.items()))
composition_cl.round(2)

Spearman(PC1, log10 cytoplasmic depth) = +0.27
Spearman(PC2, log10 cytoplasmic depth) = +0.01


Spearman(PC3, log10 cytoplasmic depth) = -0.01
Spearman(PC4, log10 cytoplasmic depth) = -0.05


Spearman(PC5, log10 cytoplasmic depth) = -0.04
                      n_cells  median_cyto_depth  median_cyto_target_counts  median_cell_area  median_dist_immune  median_dist_fibroblast  low_depth
subtype_louvain_cyto                                                                                                                                
0                      105252              174.0                       58.0              69.5                51.8                    36.6      False
1                       90657              135.0                       41.0             110.7                27.2                    40.4      False
2                       77119              148.0                       46.0              76.7                32.0                    28.3      False
3                       75521               96.0                       33.0              82.4                49.3                    42.1      False
4                       62098              128.0           

sample,Xenium_5K_BC,Xenium_5K_OC,Xenium_5K_CC,Xenium_5K_LC,Xenium_5K_Prostate,n_cells,top_sample,top_share,top_sample_ratio
subtype_louvain_cyto,,,,,,,,,
0,0.00,0.99,0.00,0.00,0.00,105252,Xenium_5K_OC,0.99,3.87
1,0.98,0.00,0.00,0.00,0.01,90657,Xenium_5K_BC,0.98,5.98
2,0.01,0.00,0.00,0.00,0.98,77119,Xenium_5K_Prostate,0.98,6.43
3,0.00,0.01,0.97,0.01,0.01,75521,Xenium_5K_CC,0.97,2.74
4,0.00,0.00,0.99,0.00,0.00,62098,Xenium_5K_CC,0.99,2.79
5,0.07,0.07,0.72,0.02,0.11,57142,Xenium_5K_CC,0.72,2.03
6,0.00,1.00,0.00,0.00,0.00,46583,Xenium_5K_OC,1.00,3.88
7,0.00,0.00,0.00,1.00,0.00,41043,Xenium_5K_LC,1.00,13.93
8,0.00,0.00,0.99,0.00,0.00,29757,Xenium_5K_CC,0.99,2.79


## 6. Differential expression (`differential_expression`)

Wilcoxon test of every pooled cluster against all other pooled cells on `layers["lognorm"]`, same thresholds as per
sample; one table per cluster in `DE/`.

In [8]:
de_dict = differential_expression(adata, paths)

Cluster 0: 72 marker genes; top: H19, EEF1G, CD47, LAPTM4B, IMPDH2, APP, SULF1, MLF1
Cluster 1: 49 marker genes; top: XBP1, TSPAN13, CCND1, HSPB8, SERPINA3, SEPTIN9, MAPKAPK2, RARA
Cluster 2: 66 marker genes; top: SORD, NKX3-1, FOXA1, EEF1G, XBP1, IL1R1, DAP, IGFBP2
Cluster 3: 41 marker genes; top: LDHA, NDRG1, SLC2A1, CDKN1A, ANXA2, APOL1, PLP2, ITGA6
Cluster 4: 64 marker genes; top: TMEM45A, CDKN1A, MALL, ANXA2, PLP2, RRM2, TPX2, NFE2L2
Cluster 5: 0 marker genes; top: 
Cluster 6: 202 marker genes; top: H19, CD47, LAPTM4B, EEF1G, SMC4, HMGA1, TOP2A, TPX2
Cluster 7: 413 marker genes; top: SLC40A1, PDIA4, EPDR1, BLVRA, CDH1, HOPX, PLS3, GRN
Cluster 8: 35 marker genes; top: CDKN1A, MALL, SAMD9, ANXA2, HOPX, BCL2L1, TGFA, PLS3
Cluster 9: 24 marker genes; top: CXCL10, CXCL9, GBP1, STAT1, UBD, IFIT3, ANXA2, MSN
Cluster 10: 60 marker genes; top: ZFAND5, TGFB3, CCN2, EEF1G, SORD, NBL1, SFRP1, SOX9
Cluster 11: 53 marker genes; top: XBP1, C4B, C4A, SERPINA3, TSPAN13, CCND1, HSPB8, DDR1
Cluster 

## 7. Annotation (`annotate`)

The per-sample rule (marker overlap, program score, Baseline, Low depth and Stroma border overrides, confidence) on
the pooled clusters, followed by the reviewed decisions of `POOLED_CONFIG["manual_labels"]`. The per-cell record
`annotation/cell_subtypes_pooled.csv` lists, for every cell, the pooled cluster and label next to the per-sample
cluster and label.

In [9]:
annotation, categories = annotate(adata, adata, de_dict, cluster_stats, scores_z, programs, label_programs, POOLED_CONFIG, paths)

cell_subtypes = pd.DataFrame({"sample": adata.obs["sample"].astype(str).values, "cell_id": adata.obs["cell_id"].values,
                              "subtype_louvain_pooled": adata.obs["subtype_louvain_cyto"].astype(str).values,
                              "subtype_label_pooled": adata.obs["subtype_label"].astype(str).values})
cell_subtypes["confidence_pooled"] = cell_subtypes["subtype_louvain_pooled"].map(annotation["confidence"].to_dict())
cell_subtypes["resolution_pooled"] = POOLED_CONFIG["res_cyto"]
for col in ["subtype_louvain_sample", "subtype_label_sample", "confidence_sample", "resolution_sample"]:
    cell_subtypes[col] = adata.obs[col].astype(str).values if col.startswith("subtype") else adata.obs[col].values
cell_subtypes.to_csv(paths["annotation"] + "cell_subtypes_pooled.csv", index=False)

16 clusters -> 9 labels; clusters to review: none
         n_cells                     label confidence  n_markers           overlap_program  overlap_genes     overlap_p             score_program  score_z
cluster                                                                                                                                                   
0         105252                  Baseline   reviewed         72           Stress response              4  1.891956e-02          Oxidative stress     0.63
1          90657                  Baseline   reviewed         49                 Apoptosis              5  3.653822e-02                 ER stress     0.77
2          77119                  Baseline   reviewed         66                 ER stress              5  4.721065e-02                 ER stress     0.48
3          75521                   Hypoxia       high         41                   Hypoxia              8  1.009987e-04                   Hypoxia     1.08
4          62098    

## 8. Subtype characteristics (`summarize_characteristics`) and saved objects

Per-label medians and IQRs of the characteristics (program scores recomputed on the pooled object; spatial context,
ssGSEA / 8-TF hypoxia and morphology from the per-sample tables) and the per-cell table, as per sample. The pooled
`.h5ad` is then rewritten with the clusters, labels (pooled and per-sample), program scores and canvas coordinates in
`obs` and the configuration in `uns["subtyping"]`, so that later stages can load one object with everything. In the
saved object the "/" of four program-score column names is replaced by "_" (`score_G2_M`, `score_EMT_stroma`,
`score_TNFA_NFKB`, `score_Complement_acute phase`), because h5py reads "/" as a group separator.

In [10]:
summary = summarize_characteristics(adata, paths, extra_cell_cols=["sample", "subtype_louvain_sample", "subtype_label_sample"])

# subtype_louvain (set by cluster()) and subtype_louvain_cyto are the same column on the pooled object: keep one in the file
adata.obs.drop(columns=[f"louvain_{res}" for res in RES_GRID] + ["subtype_louvain_cyto"], inplace=True, errors="ignore")
adata.uns["subtyping"] = {"date": datetime.date.today().isoformat(), "res_cyto": POOLED_CONFIG["res_cyto"],
                          "manual_labels": {k: v for k, v in POOLED_CONFIG["manual_labels"].items()},
                          "labels": list(categories), "notebook": "4_pooled.ipynb"}
# h5py treats "/" in a column name as a group separator, so the program-score columns "score_G2/M", "score_EMT/stroma",
# "score_TNFA/NFKB" and "score_Complement / acute phase" are stored with "_" instead (the label names themselves are
# categorical values and unaffected); the in-memory object keeps the renamed columns from here on
adata.obs.columns = [c.replace(" / ", "_").replace("/", "_") for c in adata.obs.columns]
# write to a temporary file first, so that a failed write cannot leave a truncated object behind
adata.write(pooled_path + "adata_pooled_cytoplasmic_embedded.h5ad.tmp")
os.replace(pooled_path + "adata_pooled_cytoplasmic_embedded.h5ad.tmp", pooled_path + "adata_pooled_cytoplasmic_embedded.h5ad")
adata.obs["subtype_louvain_cyto"] = adata.obs["subtype_louvain"]   # name expected by the shared functions
print(f"pooled object rewritten with {len(categories)} labels: {list(categories)}")

                          n_cells  score_Hypoxia  hypoxia_ssgsea_cyto  hypoxia_score_tf  score_Interferon  interferon_ssgsea_cyto  score_Complement / acute phase  score_S-phase  score_G2/M  score_Stress response  score_Lysosome  proximity_immune  proximity_tcell  proximity_cd8_tcell  proximity_myeloid  proximity_bcell  proximity_fibroblast  proximity_endothelial  frac_immune_30um  frac_tcell_30um  n_cells_30um  cyto_depth  cyto_target_counts  cell_area  nucleus_cell_area_ratio
subtype_label                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
S-phase                    108681   

pooled object rewritten with 9 labels: ['S-phase', 'Interferon', 'Complement / acute phase', 'Stress response', 'Hypoxia', 'Baseline', 'Stroma border', 'Myeloid border', 'Low depth']


## 9. Figures (`make_figures` + pooled-specific figures)

The per-sample figure set (UMAP by cluster and by label, PCA loadings, tissue map by label, composition, program
heatmap, marker dotplot, one map per label on the other tumor cells / immune cells / T cells, box plots, maps of
scores and proximities), with the tissue maps drawn on the tiled canvas. In addition, in `clustering/`:
`umap_sample.png` and `tsne_sample.png` (cells coloured by sample, drawn in random order so that no sample hides
another) with one panel per sample, and the stacked sample composition of every cluster and of every label.

In [11]:
make_figures(obs_all, adata, adata, de_dict, annotation, categories, scores_z, "Pooled", paths, fig_size, xy=("canvas_x", "canvas_y"))

sample_colors = {s: clr.to_hex(plt.get_cmap("tab10")(i)) for i, s in enumerate(SAMPLES)}
cell_colors = np.array([sample_colors[s] for s in adata.obs["sample"].astype(str)])
rng = np.random.default_rng(0)
order = rng.permutation(adata.shape[0])
for key, name in [("X_umap", "umap"), ("X_tsne", "tsne")]:
    if key not in adata.obsm:
        continue
    emb = adata.obsm[key]
    fig, axes = plt.subplots(1, len(SAMPLES) + 1, figsize=(5 * (len(SAMPLES) + 1), 5))
    axes[0].scatter(emb[order, 0], emb[order, 1], c=cell_colors[order], s=0.3, linewidths=0, rasterized=True)
    axes[0].set_title(f"{name.upper()}: all samples")
    for ax, s in zip(axes[1:], SAMPLES):
        is_s = (adata.obs["sample"] == s).values
        ax.scatter(emb[~is_s, 0], emb[~is_s, 1], c="lightgray", s=0.3, linewidths=0, rasterized=True)
        ax.scatter(emb[is_s, 0], emb[is_s, 1], c=[sample_colors[s]], s=0.3, linewidths=0, rasterized=True)
        ax.set_title(f"{short(s)} ({is_s.sum()} cells)")
    for ax in axes:
        ax.grid(False)
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.savefig(paths["clustering"] + f"{name}_sample.png", dpi=200, bbox_inches="tight")
    plt.close()

# stacked sample composition per cluster and per label
for groupby, fname in [("subtype_louvain_cyto", "cluster_sample_composition"), ("subtype_label", "label_sample_composition")]:
    comp = pd.crosstab(adata.obs[groupby], adata.obs["sample"], normalize="index")[SAMPLES]
    if groupby == "subtype_label":
        comp = comp.loc[categories]
    fig, ax = plt.subplots(figsize=(0.5 * len(comp) + 3, 4.5))
    bottom = np.zeros(len(comp))
    for s in SAMPLES:
        ax.bar(comp.index.astype(str), comp[s].values, bottom=bottom, color=sample_colors[s], label=short(s))
        bottom += comp[s].values
    ax.axhline(1, color="none")
    ax.grid(False)
    ax.set_ylabel("share of the cluster's cells")
    ax.set_xticklabels(comp.index.astype(str), rotation=60, ha="right")
    ax.legend(frameon=False, bbox_to_anchor=(1, 1), loc="upper left")
    ax.set_title("expected: " + ", ".join(f"{short(s)} {v:.2f}" for s, v in expected.items()), fontsize=9)
    plt.savefig(paths["clustering"] + f"{fname}.png", dpi=200, bbox_inches="tight")
    plt.close()

## 10. Comparison with the per-sample subtypes (`comparison/`)

**(1) Existence.** `label_presence.csv`: every label that occurs in any analysis, with its share of the pooled cells
and of each sample's cells under the per-sample labels, and whether the pooled clustering has it. Pooled clusters
whose dominant sample exceeds three times its expected share (`cluster_sample_composition.csv`) are listed as
sample-specific.

**(2) Correspondence.** Per sample, the contingency table of per-sample label x pooled label, row-normalised
(`correspondence_<sample>.csv`, heatmaps); the adjusted Rand index and normalised mutual information between the
two partitions of the sample's cells, at label level and at cluster level (`correspondence_scores.csv`); and the
**label agreement** (`label_agreement.csv`): for every per-sample label of every sample, the share of its cells that
carry the same pooled label, next to the best-matching pooled label and its share (which catches a program that the
pooled annotation names differently, e.g. TNFA/NFKB vs Stress response). Tiled tissue maps of the per-sample labels
and the pooled labels side by side make the agreement visible in space.

**(3) Batch mixing.** On the pooled kNN graph (the 30 nearest neighbours of every cell in PC space) the share of a
cell's neighbours that come from its own sample is compared with the sample's share of all cells:
`same_sample_ratio` = observed / expected (1 = perfectly mixed; 1 / expected share = the cell's neighbourhood is
its own sample only), together with the normalised entropy of the neighbourhood's sample composition (1 = all five
samples in proportion, 0 = one sample). Medians per sample and per pooled label go to `batch_mixing.csv`; the
one-way-ANOVA R2 of sample on each of PC1-10 (`pc_sample_variance.csv`) says how much of each component is a sample
effect. The UMAP / t-SNE panels of section 9 are the visual check.

In [12]:
comp_path = paths["comparison"]
obs = adata.obs

# (1) existence of the per-sample labels in the pooled clustering
presence = pd.DataFrame(index=[label for label in LABEL_ORDER if label in set(sample_labels) | set(categories)])
presence["pooled_pct"] = (obs["subtype_label"].value_counts(normalize=True) * 100).reindex(presence.index).fillna(0)
for s in SAMPLES:
    presence[f"{short(s)}_pct"] = (obs.loc[obs["sample"] == s, "subtype_label_sample"].value_counts(normalize=True) * 100).reindex(presence.index).fillna(0)
presence["n_samples"] = (presence[[f"{short(s)}_pct" for s in SAMPLES]] > 0).sum(axis=1)
presence["in_pooled"] = presence["pooled_pct"] > 0
presence["pooled_clusters"] = [", ".join(annotation.index[annotation["label"] == label]) for label in presence.index]
presence.round(2).to_csv(comp_path + "label_presence.csv")
sample_specific = composition_cl[composition_cl["top_sample_ratio"] > 3].copy()
sample_specific["label"] = annotation.loc[sample_specific.index, "label"]
sample_specific.round(3).to_csv(comp_path + "sample_specific_clusters.csv")
print("per-sample labels missing from the pooled clustering: " + (", ".join(presence.index[(presence["n_samples"] > 0) & ~presence["in_pooled"]]) or "none"))
print("pooled labels absent per sample: " + (", ".join(presence.index[(presence["n_samples"] == 0) & presence["in_pooled"]]) or "none"))
print(f"sample-specific pooled clusters (top sample > 3x expected): "
      + (", ".join(f"{cl} ({r.label}, {short(r.top_sample)} {r.top_share:.0%})" for cl, r in sample_specific.iterrows()) or "none"))

# (2) correspondence per sample
scores, agreement, tables = [], [], {}
for s in SAMPLES:
    o = obs[obs["sample"] == s]
    table = pd.crosstab(o["subtype_label_sample"], o["subtype_label"], normalize="index")
    table = table.reindex(index=[l for l in LABEL_ORDER if l in table.index], columns=[l for l in LABEL_ORDER if l in table.columns])
    table.round(4).to_csv(comp_path + f"correspondence_{s}.csv")
    tables[s] = table
    scores.append({"sample": s, "n_cells": len(o),
                   "ARI_label": adjusted_rand_score(o["subtype_label_sample"].astype(str), o["subtype_label"].astype(str)),
                   "NMI_label": normalized_mutual_info_score(o["subtype_label_sample"].astype(str), o["subtype_label"].astype(str)),
                   "ARI_cluster": adjusted_rand_score(o["subtype_louvain_sample"].astype(str), o["subtype_louvain_cyto"].astype(str)),
                   "NMI_cluster": normalized_mutual_info_score(o["subtype_louvain_sample"].astype(str), o["subtype_louvain_cyto"].astype(str))})
    for label in table.index:
        agreement.append({"sample": s, "label": label, "n_cells": int((o["subtype_label_sample"] == label).sum()),
                          "agreement": table.loc[label, label] if label in table.columns else 0.0,
                          "best_pooled_label": table.loc[label].idxmax(), "best_share": table.loc[label].max()})
scores = pd.DataFrame(scores).set_index("sample")
scores.round(3).to_csv(comp_path + "correspondence_scores.csv")
agreement = pd.DataFrame(agreement)
agreement.round(4).to_csv(comp_path + "label_agreement.csv", index=False)
print(scores.round(3).to_string())

fig, axes = plt.subplots(1, len(SAMPLES), figsize=(5.5 * len(SAMPLES), 6))
for ax, s in zip(axes, SAMPLES):
    sns.heatmap(tables[s], cmap="Blues", vmin=0, vmax=1, annot=True, fmt=".2f", annot_kws={"size": 6}, cbar=False, ax=ax)
    ax.grid(False)
    ax.set_title(f"{short(s)}: ARI {scores.loc[s, 'ARI_label']:.2f}, NMI {scores.loc[s, 'NMI_label']:.2f}")
    ax.set_xlabel("pooled label")
    ax.set_ylabel("per-sample label" if s == SAMPLES[0] else "")
plt.tight_layout()
plt.savefig(comp_path + "correspondence_heatmaps.png", dpi=200, bbox_inches="tight")
plt.close()

fig, ax = plt.subplots(figsize=(0.8 * len(presence) + 2, 4.5))
for i, s in enumerate(SAMPLES):
    d = agreement[agreement["sample"] == s].set_index("label").reindex(presence.index)
    ax.plot(np.arange(len(presence)), d["agreement"].values, "o", color=sample_colors[s], label=short(s))
ax.axhline(0.5, color="gray", lw=0.8, ls="--")
ax.grid(False)
ax.set_xticks(np.arange(len(presence)))
ax.set_xticklabels(presence.index, rotation=60, ha="right")
ax.set_ylabel("share of cells with the same pooled label")
ax.set_ylim(0, 1.02)
ax.legend(frameon=False)
plt.savefig(comp_path + "label_agreement.png", dpi=200, bbox_inches="tight")
plt.close()

# per-sample labels vs pooled labels on the canvas
fig, axes = plt.subplots(1, 2, figsize=(2 * fig_size[0], fig_size[1]))
for ax, col, title in zip(axes, ["subtype_label_sample", "subtype_label"], ["per-sample labels", "pooled labels"]):
    ax.scatter(obs["canvas_x"], obs["canvas_y"], c=obs[col].astype(str).map(LABEL_COLORS).values, s=0.4, linewidths=0, rasterized=True)
    label_canvas(ax)
    ax.grid(False)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
handles = [plt.Line2D([], [], marker="o", ls="", color=LABEL_COLORS[l], label=l) for l in LABEL_ORDER if l in set(sample_labels) | set(categories)]
axes[1].legend(handles=handles, frameon=False, bbox_to_anchor=(1, 1), loc="upper left", fontsize=9)
plt.savefig(comp_path + "scatter_labels_sample_vs_pooled.png", dpi=200, bbox_inches="tight")
plt.close()

# (3) batch mixing on the pooled kNN graph
D = adata.obsp["distances"].tocsr()
codes = obs["sample"].cat.codes.values
rows = np.repeat(np.arange(D.shape[0]), np.diff(D.indptr))
n_neighbors = np.diff(D.indptr)
same = (codes[rows] == codes[D.indices]).astype(float)
same_frac = np.bincount(rows, weights=same, minlength=D.shape[0]) / np.maximum(n_neighbors, 1)
counts = sparse.csr_matrix((np.ones(len(rows)), (rows, codes[D.indices])), shape=(D.shape[0], len(SAMPLES))).toarray()
p = counts / np.maximum(counts.sum(axis=1, keepdims=True), 1)
with np.errstate(divide="ignore", invalid="ignore"):
    entropy = np.clip(-np.nansum(np.where(p > 0, p * np.log(p), 0), axis=1) / np.log(len(SAMPLES)), 0, 1)
obs["same_sample_frac"] = same_frac
obs["same_sample_ratio"] = same_frac / expected.loc[obs["sample"].astype(str)].values
obs["neighbourhood_entropy"] = entropy
mixing = pd.concat([obs.groupby("sample", observed=True)[["same_sample_frac", "same_sample_ratio", "neighbourhood_entropy"]].median().assign(group="sample"),
                    obs.groupby("subtype_label", observed=True)[["same_sample_frac", "same_sample_ratio", "neighbourhood_entropy"]].median().assign(group="label")])
mixing["expected_frac"] = [expected.get(i, np.nan) for i in mixing.index]
mixing["max_ratio"] = [1 / expected[i] if i in expected.index else np.nan for i in mixing.index]
mixing.round(3).to_csv(comp_path + "batch_mixing.csv")
print(mixing.round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
sns.boxplot(data=obs, x="sample", y="same_sample_ratio", order=SAMPLES, palette=sample_colors, showfliers=False, ax=axes[0])
axes[0].axhline(1, color="gray", lw=0.8, ls="--")
axes[0].set_title("same-sample neighbours / expected (1 = mixed)")
axes[0].set_xticklabels([short(s) for s in SAMPLES])
sns.boxplot(data=obs, x="sample", y="neighbourhood_entropy", order=SAMPLES, palette=sample_colors, showfliers=False, ax=axes[1])
axes[1].set_title("neighbourhood sample entropy (1 = all samples in proportion)")
axes[1].set_xticklabels([short(s) for s in SAMPLES])
sns.boxplot(data=obs, x="subtype_label", y="same_sample_ratio", order=categories, palette=LABEL_COLORS, showfliers=False, ax=axes[2])
axes[2].axhline(1, color="gray", lw=0.8, ls="--")
axes[2].set_title("same-sample ratio per pooled label")
axes[2].set_xticklabels(categories, rotation=60, ha="right")
for ax in axes:
    ax.grid(False)
    ax.set_xlabel("")
plt.tight_layout()
plt.savefig(comp_path + "batch_mixing.png", dpi=200, bbox_inches="tight")
plt.close()

# variance of the PCs explained by sample (one-way ANOVA R2)
pcs = pd.DataFrame(adata.obsm["X_pca"][:, :10], columns=[f"PC{i + 1}" for i in range(10)], index=obs.index)
pcs["sample"] = obs["sample"].values
total_ss = ((pcs.iloc[:, :10] - pcs.iloc[:, :10].mean()) ** 2).sum()
group_means = pcs.groupby("sample", observed=True).mean()
group_sizes = pcs.groupby("sample", observed=True).size()
between_ss = ((group_means - pcs.iloc[:, :10].mean()) ** 2).mul(group_sizes, axis=0).sum()
pc_var = pd.DataFrame({"variance_ratio": adata.uns["pca"]["variance_ratio"][:10], "sample_R2": (between_ss / total_ss).values}, index=total_ss.index)
pc_var.round(4).to_csv(comp_path + "pc_sample_variance.csv")
print(pc_var.round(3).T.to_string())

per-sample labels missing from the pooled clustering: G2/M, Lysosome, Cholesterol synthesis, EMT/stroma, ER stress, MYC targets
pooled labels absent per sample: none
sample-specific pooled clusters (top sample > 3x expected): 0 (Baseline, OC 99%), 1 (Baseline, BC 98%), 2 (Baseline, Prostate 98%), 6 (S-phase, OC 100%), 7 (Baseline, LC 100%), 10 (Stroma border, Prostate 89%), 11 (Complement / acute phase, BC 85%), 13 (Stroma border, OC 89%)


                    n_cells  ARI_label  NMI_label  ARI_cluster  NMI_cluster
sample                                                                     
Xenium_5K_BC         102180      0.063      0.108        0.020        0.095
Xenium_5K_OC         160250      0.222      0.249        0.101        0.231
Xenium_5K_CC         221355      0.180      0.280        0.197        0.297
Xenium_5K_LC          44624      0.001      0.057        0.014        0.087
Xenium_5K_Prostate    95429      0.118      0.196        0.040        0.201


                          same_sample_frac  same_sample_ratio  neighbourhood_entropy   group  expected_frac  max_ratio
Xenium_5K_BC                         1.000              6.105                  0.000  sample          0.164      6.105
Xenium_5K_OC                         1.000              3.893                  0.000  sample          0.257      3.893
Xenium_5K_CC                         1.000              2.818                  0.000  sample          0.355      2.818
Xenium_5K_LC                         1.000             13.980                  0.000  sample          0.072     13.980
Xenium_5K_Prostate                   1.000              6.537                  0.000  sample          0.153      6.537
S-phase                              1.000              2.818                  0.000   label            NaN        NaN
Interferon                           1.000              2.818                  0.000   label            NaN        NaN
Complement / acute phase             1.000      

                  PC1    PC2    PC3    PC4    PC5    PC6    PC7    PC8    PC9   PC10
variance_ratio  0.018  0.012  0.010  0.008  0.005  0.004  0.003  0.003  0.003  0.003
sample_R2       0.722  0.688  0.498  0.651  0.082  0.006  0.006  0.005  0.012  0.011


## 11. Summary

Criteria used for the three answers (`comparison/summary.txt`):

1. A per-sample label counts as **recovered** when the pooled clustering has it and, over the samples where the
   per-sample analysis found it, the median label agreement is at least 0.5 (half of its cells carry the same pooled
   label). The artifact classes (Baseline, Stroma border, Myeloid border, Low depth) are reported but not counted.
2. **Correspondence** is summarised by the per-sample ARI / NMI at label level.
3. **Mixing** is good when the median same-sample ratio of every sample is below 2 (less than twice the expected share
   of own-sample neighbours) and no PC among the first five has a sample R2 above 0.2.

In [13]:
artifact_labels = {"Baseline", "Stroma border", "Myeloid border", "Low depth"}
n_str = ", ".join(f"{short(s)} {n}" for s, n in adata.obs["sample"].value_counts().loc[SAMPLES].items())
lines = [f"Pooled subtyping of {adata.shape[0]} tumor cells ({n_str}), "
         f"resolution {POOLED_CONFIG['res_cyto']}, {len(annotation)} clusters -> {len(categories)} labels", ""]

lines.append("(1) Existence of the per-sample subtypes in the pooled clustering")
recovered, lost = [], []
for label in presence.index:
    if label in artifact_labels or presence.loc[label, "n_samples"] == 0:
        continue
    a = agreement[(agreement["label"] == label)]
    med = a["agreement"].median()
    where = ", ".join(f"{short(r.sample)} {r.agreement:.2f}" for r in a.itertuples())
    ok = presence.loc[label, "in_pooled"] and med >= 0.5
    (recovered if ok else lost).append(label)
    lines.append(f"  {label:26s} per-sample in {int(presence.loc[label, 'n_samples'])} sample(s); pooled {presence.loc[label, 'pooled_pct']:.1f}% "
                 f"({'present' if presence.loc[label, 'in_pooled'] else 'absent'}); agreement {where} -> {'recovered' if ok else 'not recovered'}")
new_pooled = [l for l in categories if presence.loc[l, "n_samples"] == 0]
lines.append(f"  recovered: {', '.join(recovered) or 'none'}")
lines.append(f"  not recovered: {', '.join(lost) or 'none'}")
lines.append(f"  pooled-only labels: {', '.join(new_pooled) or 'none'}")
specific_str = ", ".join(f"{cl} ({annotation.loc[cl, 'label']})" for cl in sample_specific.index) or "none"
lines.append(f"  sample-specific pooled clusters: {specific_str}")
lines.append("")
lines.append("(2) Correspondence of per-sample and pooled labels within each sample")
for s in SAMPLES:
    lines.append(f"  {short(s):9s} ARI {scores.loc[s, 'ARI_label']:.2f}, NMI {scores.loc[s, 'NMI_label']:.2f} (labels); "
                 f"ARI {scores.loc[s, 'ARI_cluster']:.2f}, NMI {scores.loc[s, 'NMI_cluster']:.2f} (clusters)")
lines.append("")
lines.append("(3) Batch mixing in the pooled embedding")
for s in SAMPLES:
    lines.append(f"  {short(s):9s} median same-sample ratio {mixing.loc[s, 'same_sample_ratio']:.2f} (1 = mixed, {mixing.loc[s, 'max_ratio']:.1f} = separated), "
                 f"neighbourhood entropy {mixing.loc[s, 'neighbourhood_entropy']:.2f}")
lines.append("  sample R2 of PC1-5: " + ", ".join(f"{pc} {r:.2f}" for pc, r in pc_var['sample_R2'].head(5).items()))
mixed = (mixing.loc[SAMPLES, "same_sample_ratio"] < 2).all() and (pc_var["sample_R2"].head(5) < 0.2).all()
lines.append(f"  -> {'no strong batch effect' if mixed else 'sample effect present'} by the criteria above")
summary_text = "\n".join(lines)
open(comp_path + "summary.txt", "w").write(summary_text + "\n")
print(summary_text)

Pooled subtyping of 623838 tumor cells (BC 102180, OC 160250, CC 221355, LC 44624, Prostate 95429), resolution 1.0, 16 clusters -> 9 labels

(1) Existence of the per-sample subtypes in the pooled clustering
  S-phase                    per-sample in 5 sample(s); pooled 17.4% (present); agreement BC 0.01, OC 0.73, CC 0.58, LC 0.03, Prostate 0.03 -> not recovered
  G2/M                       per-sample in 3 sample(s); pooled 0.0% (absent); agreement BC 0.00, OC 0.00, CC 0.00 -> not recovered
  Interferon                 per-sample in 4 sample(s); pooled 2.4% (present); agreement BC 0.03, OC 0.24, CC 0.59, LC 0.03 -> not recovered
  Complement / acute phase   per-sample in 3 sample(s); pooled 0.9% (present); agreement BC 0.35, LC 0.00, Prostate 0.15 -> not recovered
  Stress response            per-sample in 4 sample(s); pooled 4.8% (present); agreement BC 0.00, OC 0.00, CC 0.70, Prostate 0.00 -> not recovered
  Lysosome                   per-sample in 2 sample(s); pooled 0.0% (absent); a